# Time Series Council Demo

**A Natural Language Interface for Time Series Analysis**

This notebook demonstrates how to analyze time series data using:
- **Multiple LLM providers** (Gemini, Claude, OpenAI, DeepSeek, Qwen)
- **Multiple forecasting models** (Moirai2, Chronos, TimesFM, Ti-Rex, Lag-Llama, LLM-based)
- **Multiple anomaly detectors** (Z-score, MAD, Isolation Forest, LOF, LSTM-VAE, Merlion, PyOD, Moirai, RuleDetector)
- **9 analysis tools** (forecast, describe, anomalies, decompose, compare, what-if, sensitivity, backtest, compare-periods)
- **Council modes** for multi-perspective analysis
- **Detection memory** for stateful anomaly detection
- **Progress tracking** with real-time callbacks

---

## 📦 Setup & Installation

In [ ]:
import sys
import os
from IPython.display import HTML, display

# Ensure the package is importable (if running from the examples/ directory)
if os.path.basename(os.getcwd()) == 'examples':
    os.chdir('..')
    sys.path.insert(0, 'src')

print(f"Working directory: {os.getcwd()}")

## 🔧 Initialize with Factory Functions

The library uses factory functions to create LLM providers, forecasters, and detectors.
This makes it easy to swap components.

In [ ]:
from timeseries_council import Orchestrator, Config
from timeseries_council.providers import create_provider
from timeseries_council.forecasters import create_forecaster
from timeseries_council.detectors import create_detector
from pathlib import Path
import pandas as pd

# Configuration
CSV_PATH = str(Path("data/sample_sales.csv").absolute())
TARGET_COL = "sales"

# Create LLM provider (reads API key from environment or config.yaml)
api_key = os.getenv("GEMINI_API_KEY")
provider = create_provider("gemini", api_key)  # or "anthropic", "openai", "deepseek", "qwen"

# Create forecaster and detector (optional - orchestrator has defaults)
forecaster = create_forecaster("moirai")    # or "chronos", "timesfm", "tirex", "lag_llama", "llm"
detector = create_detector("zscore")        # or "mad", "isolation_forest", "lof", "lstm_vae", "merlion", "pyod"

# Initialize orchestrator
orchestrator = Orchestrator(
    llm_provider=provider,
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    forecaster=forecaster,
    detector=detector,
)

print("✅ Orchestrator ready!")

## 📊 View the Sample Data

In [ ]:
# Load and display the data
df = pd.read_csv(CSV_PATH, parse_dates=[0], index_col=0)
print(f"📈 Data shape: {df.shape}")
print(f"📅 Date range: {df.index[0]} to {df.index[-1]}")
df.head(10)

In [ ]:
# Plot the data
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.plot(df.index, df[TARGET_COL], linewidth=0.8)
plt.title('Sample Sales Data', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# 💬 Natural Language Queries

Ask questions in plain English — the LLM selects the right tool automatically.

### Example 1: Describe the Data

In [ ]:
response = orchestrator.chat("Describe the sales data and what trends you see")
display(HTML(response))

### Example 2: Forecast Future Values

In [ ]:
response = orchestrator.chat("Forecast sales for the next 7 days, using Moirai")
print(response)

### Example 3: Detect Anomalies

In [ ]:
response = orchestrator.chat("Are there any unusual spikes or drops in the sales data?")
display(HTML(response))

### Example 4: Decompose Time Series

In [ ]:
response = orchestrator.chat("Decompose the sales data and show me the seasonal patterns")
display(HTML(response))

### Example 5: What-If Scenario Analysis

In [ ]:
response = orchestrator.chat("What if sales increase by 20%?")
print(response)

### Example 6: Compare Time Periods

In [ ]:
response = orchestrator.chat("Compare sales in March versus September")
print(response)

### Example 7: Backtest Forecasts

In [ ]:
response = orchestrator.chat("Backtest the forecast for September and show me how accurate it was")
print(response)

### Example 8: General Question (Direct Answer)

In [ ]:
response = orchestrator.chat("What kind of analysis can you do?")
display(HTML(response))

---

# 🔧 Direct Tool Usage (Without LLM)

You can also call the tools directly for programmatic access.

### Direct Forecast

In [ ]:
from timeseries_council.tools import run_forecast

result = run_forecast(CSV_PATH, TARGET_COL, horizon=14)

if result['success']:
    print("📈 14-Day Forecast:")
    for ts, val, unc in zip(result['timestamps'], result['forecast'], result['uncertainty']):
        print(f"  {ts[:10]}: {val:.2f} (±{unc:.2f})")
else:
    print(f"❌ Error: {result['error']}")

### Direct Statistics

In [ ]:
from timeseries_council.tools import describe_series

stats = describe_series(CSV_PATH, TARGET_COL)

if stats['success']:
    print("📊 Data Statistics:")
    print(f"  Count: {stats['count']} data points")
    print(f"  Mean: {stats['mean']:.2f}")
    print(f"  Std Dev: {stats['std']:.2f}")
    print(f"  Range: {stats['min']:.2f} - {stats['max']:.2f}")
    print(f"  Trend: {stats['trend']}")
    print(f"  Total Change: {stats['total_change_pct']:.1f}%")

### Direct Anomaly Detection

In [ ]:
from timeseries_council.tools import detect_anomalies

anomalies = detect_anomalies(CSV_PATH, TARGET_COL, sensitivity=2.0)

if anomalies['success']:
    print(f"🔍 Found {anomalies['anomaly_count']} anomalies:")
    for a in anomalies['anomalies'][:5]:  # Show first 5
        print(f"  {a['timestamp'][:10]}: {a['value']:.2f} (type={a.get('type', 'N/A')}, score={a.get('score', 'N/A')})")

### Direct Decomposition

In [ ]:
from timeseries_council.tools import decompose_series

decomp = decompose_series(CSV_PATH, TARGET_COL)

if decomp['success']:
    print("📉 Decomposition Results:")
    print(f"  Trend change: {decomp['trend_change_pct']:.1f}%")
    print(f"  Seasonal amplitude: {decomp['seasonal_amplitude']:.2f}")
    print(f"  Residual std: {decomp['residual_std']:.2f}")

### Direct Period Comparison

In [ ]:
from timeseries_council.tools import compare_periods

comparison = compare_periods(CSV_PATH, TARGET_COL, period_type="month")

if comparison['success']:
    print("📊 Monthly Period Comparison:")
    for period in comparison.get('periods', [])[:3]:
        print(f"  {period.get('label', 'N/A')}: mean={period.get('mean', 0):.2f}")

### Direct Backtesting

In [ ]:
from timeseries_council.tools import backtest_forecast

bt = backtest_forecast(CSV_PATH, TARGET_COL, target_month="september")

if bt.get('success'):
    print("🎯 Backtest Results (September):")
    metrics = bt.get('metrics', {})
    if metrics:
        print(f"  MAE: {metrics.get('mae', 'N/A')}")
        print(f"  MAPE: {metrics.get('mape', 'N/A')}")
        print(f"  RMSE: {metrics.get('rmse', 'N/A')}")

---

# 🧠 Detection Memory

The anomaly detection pipeline supports **stateful memory** — pass historical context
(baseline statistics, expected ranges, domain knowledge) so detectors make more informed decisions.

In [ ]:
from timeseries_council.types import DetectionMemory

# Create memory with known-normal baseline
memory = DetectionMemory(
    baseline_stats={"mean": 1100.0, "std": 150.0, "median": 1080.0},
    expected_range=[800, 1500],
    context="Holiday season — expect higher variability",
)

print(f"Memory baseline mean: {memory.baseline_stats['mean']}")
print(f"Expected range: {memory.expected_range}")
print(f"Context: {memory.context}")
print()
print("When using the Orchestrator, detection memory is automatically accumulated")
print("across calls — baseline stats from each detection run carry forward.")

---

# 📡 Progress Tracking

Track analysis progress in real-time using callbacks.

In [ ]:
from timeseries_council.types import ProgressStage

def progress_callback(stage: ProgressStage, message: str, progress: float):
    bar = '█' * int(progress * 20) + '░' * (20 - int(progress * 20))
    print(f"  [{stage.value:20s}] {bar} {progress:.0%} — {message}")

# Create orchestrator with progress tracking
orchestrator_with_progress = Orchestrator(
    llm_provider=provider,
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    forecaster=forecaster,
    detector=detector,
    progress_callback=progress_callback,
)

print("Running analysis with progress tracking...\n")
response = orchestrator_with_progress.chat("Describe the sales data")
print(f"\n📝 Result:\n{response[:200]}...")

---

# 🏛️ Council Mode

Get multi-perspective analysis from different analyst roles.

### Standard Council Mode

Three specialized roles provide different perspectives:
- **Quantitative Analyst**: Statistical significance, confidence intervals
- **Risk Analyst / Domain Expert**: Risks, uncertainties, domain context
- **Opportunity Analyst / Business Explainer**: Business implications, actionable insights

In [ ]:
# Council mode provides richer, multi-perspective analysis
response = orchestrator.chat_with_council(
    "Analyze the sales data and tell me what risks I should know about"
)
print(response)

In [ ]:
# Try council mode with a forecast question
response = orchestrator.chat_with_council(
    "What will sales look like next week and what should I prepare for?"
)
print(response)

### Advanced Council Mode (Karpathy-style)

The Advanced Council runs a 3-stage deliberation process:
1. **Stage 1**: Multiple LLMs provide initial responses
2. **Stage 2**: Models rank each other's responses (anonymized)
3. **Stage 3**: Chairman synthesizes the final answer

This requires multiple LLM providers with their respective API keys.

In [ ]:
# Advanced Council example (requires multiple API keys)
# Uncomment and configure with your available providers:

# from timeseries_council.council import AdvancedCouncil
#
# council = AdvancedCouncil(
#     council_providers={
#         "gemini": create_provider("gemini", os.getenv("GEMINI_API_KEY")),
#         "claude": create_provider("anthropic", os.getenv("ANTHROPIC_API_KEY")),
#         "gpt4":  create_provider("openai", os.getenv("OPENAI_API_KEY")),
#     },
#     chairman_name="claude"
# )
#
# result = council.run_sync(
#     user_query="What will sales be next month?",
#     context="Historical data context..."
# )
# print(result)

print("ℹ️  Advanced Council requires multiple LLM API keys.")
print("    Supported providers: gemini, anthropic, openai, deepseek, qwen")

---

# 🔄 Using pd.Series Directly

You can also use the library with `pd.Series` directly, without CSV files.

In [ ]:
from timeseries_council.tools import detect_anomalies, run_forecast

# Create a sample series programmatically
series = pd.Series(
    [10.0, 11.2, 10.8, 50.0, 10.9, 11.5, 10.3],
    index=pd.date_range("2024-01-01", periods=7, freq="D"),
)

print("📊 Sample Series:")
print(series)
print()

# Detect anomalies directly from series
result = detect_anomalies(series=series)
if result['success']:
    print(f"🔍 Found {result['anomaly_count']} anomalies")
    for a in result['anomalies']:
        print(f"  {a['timestamp'][:10]}: {a['value']} ({a['type']})")

# You can also create an Orchestrator with pd.Series
# orchestrator_series = Orchestrator(
#     llm_provider=provider,
#     data=series,
# )
# response = orchestrator_series.chat("Are there any anomalies?")

---

## ✅ Summary of Available Tools

| Tool | Description | Trigger Phrases |
|------|-------------|----------------|
| `run_forecast` | Predict future values using AI models | "predict", "forecast", "next N days" |
| `describe_series` | Statistical summary with trend analysis | "describe", "trend", "statistics" |
| `detect_anomalies` | Find anomalies using ML or rule-based detectors | "anomalies", "spikes", "unusual" |
| `decompose_series` | Trend + seasonal + residual decomposition | "decompose", "seasonality", "pattern" |
| `compare_series` | Compare multiple columns/series | "compare columns", "correlation" |
| `what_if_simulation` | Scenario analysis — simulate changes | "what if", "scenario", "impact" |
| `sensitivity_analysis` | Parameter sensitivity testing | "sensitivity", "parameter" |
| `compare_periods` | Compare months, quarters, or other periods | "compare March and April", "which month" |
| `backtest_forecast` | Test forecast accuracy on historical data | "backtest", "accuracy", "validate" |

**Council Mode:** Use `orchestrator.chat_with_council()` for multi-perspective analysis.

---

## 📋 Available Models

### Forecasters

| Name | Description | Dependencies |
|------|-------------|--------------|
| `moirai` | Salesforce Moirai2 via uni2ts | `uni2ts`, `gluonts`, `torch` |
| `chronos` | Amazon Chronos | `chronos-forecasting` |
| `timesfm` | Google TimesFM | `timesfm` |
| `tirex` | Ti-Rex | `tirex-ts` |
| `lag_llama` | Lag-Llama | `lag-llama` |
| `llm` | LLM-based forecasting | LLM provider |
| `zscore_baseline` | Simple statistical baseline | Built-in |

### Detectors

| Name | Description | Dependencies |
|------|-------------|--------------|
| `zscore` | Z-score detection | Built-in |
| `mad` | Median Absolute Deviation | Built-in |
| `ruledetector` | Rule-based Anomaly Detector | Built-in |
| `isolation_forest` | Isolation Forest | `scikit-learn` |
| `lof` | Local Outlier Factor | `scikit-learn` |
| `moirai` | Moirai2 back-prediction | `uni2ts`, `gluonts`, `torch` |
| `merlion` | Merlion ensemble | `salesforce-merlion` |
| `lstm_vae` | LSTM Variational Autoencoder | `torch` |
| `pyod` | PyOD detectors (ECOD, COPOD, etc.) | `pyod` |
| `llm` | LLM-based detection | LLM provider |

### LLM Providers

| Name | Environment Variable |
|------|---------------------|
| `gemini` | `GEMINI_API_KEY` |
| `anthropic` | `ANTHROPIC_API_KEY` |
| `openai` | `OPENAI_API_KEY` |
| `deepseek` | `DEEPSEEK_API_KEY` |
| `qwen` | `DASHSCOPE_API_KEY` |

---

## 📐 Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                     User Question                           │
│              "Forecast sales for next week"                 │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   LLM Provider                              │
│   (Gemini / Claude / GPT / DeepSeek / Qwen)                │
│     Interprets question → Decides tool to call              │
│     Output: {"tool": "run_forecast", "args": {...}}         │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   Orchestrator                              │
│        Parses JSON → Calls appropriate tool                 │
│        Manages detection memory across calls                │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   Tools (9 available)                       │
│   • run_forecast       → Moirai2 / Chronos / TimesFM / ... │
│   • describe_series    → Statistics + trend analysis        │
│   • detect_anomalies   → Z-score / IForest / LOF / ...     │
│   • decompose_series   → Trend + seasonal + residual       │
│   • compare_series     → Multi-column comparison           │
│   • what_if_simulation → Scenario analysis                 │
│   • sensitivity_analysis → Parameter testing               │
│   • compare_periods    → Month/quarter comparison          │
│   • backtest_forecast  → Historical accuracy testing       │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   LLM Provider                              │
│          Summarizes results in natural language             │
│     (Optional: Council mode for multi-perspective)          │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   User Response                             │
│    "The forecast for next week shows sales around..."       │
└─────────────────────────────────────────────────────────────┘
```

---

## 🎯 Key Points

1. **Multi-Provider LLM Support** — Gemini, Claude, OpenAI, DeepSeek, Qwen
2. **Pluggable Forecasters** — Moirai2, Chronos, TimesFM, Ti-Rex, Lag-Llama, LLM, baseline
3. **Pluggable Detectors** — Z-score, MAD, Isolation Forest, LOF, LSTM-VAE, Merlion, PyOD, rules
4. **9 Analysis Tools** — forecast, describe, detect, decompose, compare, what-if, sensitivity, backtest, compare-periods
5. **Council Mode** — Multi-perspective analysis from specialized roles
6. **Advanced Council** — Karpathy-style 3-stage deliberation with peer ranking
7. **Detection Memory** — Stateful anomaly detection with baseline context
8. **Progress Tracking** — Real-time callbacks for UI integration
9. **Web UI & CLI** — FastAPI interface and command-line tools
10. **pd.Series Support** — Use directly without CSV files